# NHANES project about Periodontal disease and Geriatric Nutrition Risk Index (GNRI): descriptive and regression analysis
> This notebook has the purpose to collect all the analysis on Nhanes dataset for a medical paper project 

Requirements and Information:
1. Nhanes dataset from 2009/10 to 2013/14
2. Outcome:
    - Geriatric Nutrition Risk Index (GNRI)
3. Exposure:
    - calculated using Periodontal Exam (OHXPER_H) and CDC/AAP criteria (Eke et al., 2012)
    - consists of 3 categories: None/Mild, Moderate, Severe
4. Confounding Variables:
    - Gender (RIAGENDR)
    - Age at screening (RIDAGEYR)
    - Race (RIDRETH1)
    - Education	(DMDEDUC2)
    - Poverty income ratio (INDFMPIR)
    - Smoking status (SMQ020)
    - Alchool intake (ALQ101)
5. Mediators:
    - Heart failure	(RIDRETH1)  
    - Coronary heart disease (MCQ160b)
    - Stroke (MCQ160c)
    - Liver disease	(MCQ160o)
    - Cancer (MCQ220)
    - Diabetes (DIQ010)
    - High blood pressure (BPQ020)
6. Age => 60

## Import Libraries

In [3]:
library(haven)
library(nhanesA)
library(survey)
library(MASS)
library(dplyr)
library(tidyr)
library(tidyverse)
library(ggplot2)
library(readr)
library(flextable)
library(officer)
library(nnet)
library(broom)
library(ggplot2)
library(patchwork)

Loading required package: grid

Loading required package: Matrix

Loading required package: survival


Attaching package: 'survey'


The following object is masked from 'package:graphics':

    dotchart



Attaching package: 'dplyr'


The following object is masked from 'package:MASS':

    select


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union



Attaching package: 'tidyr'


The following objects are masked from 'package:Matrix':

    expand, pack, unpack


-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v forcats   1.0.0     v readr     2.1.5
v ggplot2   3.5.1     v stringr   1.5.1
v lubridate 1.9.4     v tibble    3.2.1
v purrr     1.0.2     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x tidyr::expand() masks Matrix::expand()
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stat

## Configurations

In [5]:
path_to_data_09_10 <- "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/2009_10/"
path_to_data_11_12 <- "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/2011_12/"
path_to_data_13_14 <- "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/2013_14/"

## Load Dataset & Feature Selection

In [ ]:
# Datasets for 2009/10 period

demo_09_10 <- read_xpt(file.path(path_to_data_09_10, "DEMO_F.xpt"))

demo_09_10_selected <- demo_09_10 %>%
  select(SEQN, RIAGENDR, RIDAGEYR, RIDRETH1, DMDEDUC2, INDFMPIR)

alcohol_09_10 <- read_xpt(file.path(path_to_data_09_10, "ALQ_F.xpt"))

alcohol_09_10_selected <- alcohol_09_10 %>%
  select(SEQN, ALQ101)

smoking_09_10 <- read_xpt(file.path(path_to_data_09_10, "SMQ_F.xpt.txt"))

smoking_09_10_selected <- smoking_09_10 %>%
    select(SEQN, SMQ020)

med_conditions_09_10 <- read_xpt(file.path(path_to_data_09_10, "MCQ_F.xpt"))

med_conditions_09_10_selected <- med_conditions_09_10 %>%
    select(SEQN, MCQ160B, MCQ160C, MCQ160D, MCQ160E, MCQ160F, MCQ160L, MCQ220)

blood_pressure_09_10 <- read_xpt(file.path(path_to_data_09_10, "BPQ_F.xpt"))

blood_pressure_09_10_selected <- blood_pressure_09_10 %>%
    select(SEQN, BPQ020)


diabetes_09_10 <- read_xpt(file.path(path_to_data_09_10, "DIQ_F.xpt"))

diabetes_09_10_selected <- diabetes_09_10 %>%
    select(SEQN, DIQ010)


periodontal_09_10 <- read_xpt(file.path(path_to_data_09_10, "OHXPER_F.xpt.txt"))

selected_cols <- colnames(periodontal_09_10)[grepl("^OHX\\d{2}(PC|LA)[A-Z]$", colnames(periodontal_09_10))]

periodontal_09_10_selected <- periodontal_09_10 %>%
    select(SEQN, all_of(selected_cols))


albumin_09_10 <- read_xpt(file.path(path_to_data_09_10, "BIOPRO_F.xpt.txt"))

albumin_09_10_selected <- albumin_09_10 %>%
    select(SEQN, LBDSALSI)

w_h_09_10 <- read_xpt(file.path(path_to_data_09_10, "BMX_F.xpt"))

w_h_09_10_selected <- w_h_09_10 %>%
    select(SEQN, BMXWT, BMXHT)

In [ ]:
# Datasets for 2011/12 period

demo_11_12 <- read_xpt(file.path(path_to_data_11_12, "DEMO_G.xpt.txt"))

demo_11_12_selected <- demo_11_12 %>%
  select(SEQN, RIAGENDR, RIDAGEYR, RIDRETH1, DMDEDUC2, INDFMPIR)

alcohol_11_12 <- read_xpt(file.path(path_to_data_11_12, "ALQ_G.xpt.txt"))

alcohol_11_12_selected <- alcohol_11_12 %>%
  select(SEQN, ALQ101)


smoking_11_12 <- read_xpt(file.path(path_to_data_11_12, "SMQ_G.xpt.txt"))

smoking_11_12_selected <- smoking_11_12 %>%
    select(SEQN, SMQ020)


med_conditions_11_12 <- read_xpt(file.path(path_to_data_11_12, "MCQ_G.xpt.txt"))

med_conditions_11_12_selected <- med_conditions_11_12 %>%
    select(SEQN, MCQ160B, MCQ160C, MCQ160D, MCQ160E, MCQ160F, MCQ160L, MCQ220)


blood_pressure_11_12 <- read_xpt(file.path(path_to_data_11_12, "BPQ_G.xpt.txt"))

blood_pressure_11_12_selected <- blood_pressure_11_12 %>%
    select(SEQN, BPQ020)


diabetes_11_12 <- read_xpt(file.path(path_to_data_11_12, "DIQ_G.xpt.txt"))

diabetes_11_12_selected <- diabetes_11_12 %>%
    select(SEQN, DIQ010)


periodontal_11_12 <- read_xpt(file.path(path_to_data_11_12, "OHXPER_G.xpt.txt"))

selected_cols <- colnames(periodontal_11_12)[grepl("^OHX\\d{2}(PC|LA)[A-Z]$", colnames(periodontal_11_12))]

periodontal_11_12_selected <- periodontal_11_12 %>%
    select(SEQN, all_of(selected_cols))


albumin_11_12 <- read_xpt(file.path(path_to_data_11_12, "BIOPRO_G.xpt.txt"))

albumin_11_12_selected <- albumin_11_12 %>%
    select(SEQN, LBDSALSI)

w_h_11_12 <- read_xpt(file.path(path_to_data_11_12, "BMX_G.xpt.txt"))

w_h_11_12_selected <- w_h_11_12 %>%
    select(SEQN, BMXWT, BMXHT)

In [ ]:
# Datasets for 2013/14 period

demo_13_14 <- read_xpt(file.path(path_to_data_13_14, "DEMO_H.xpt.txt"))

demo_13_14_selected <- demo_13_14 %>%
    select(SEQN, RIAGENDR, RIDAGEYR, RIDRETH1, DMDEDUC2, INDFMPIR)


alcohol_13_14 <- read_xpt(file.path(path_to_data_13_14, "ALQ_H.xpt.txt"))

alcohol_13_14_selected <- alcohol_13_14 %>%
    select(SEQN, ALQ101)


smoking_13_14 <- read_xpt(file.path(path_to_data_13_14, "SMQ_H.xpt.txt"))

smoking_13_14_selected <- smoking_13_14 %>%
    select(SEQN, SMQ020)


med_conditions_13_14 <- read_xpt(file.path(path_to_data_13_14, "MCQ_H.xpt.txt"))

med_conditions_13_14_selected <- med_conditions_13_14 %>%
    select(SEQN, MCQ160B, MCQ160C, MCQ160D, MCQ160E, MCQ160F, MCQ160L, MCQ220)


blood_pressure_13_14 <- read_xpt(file.path(path_to_data_13_14, "BPQ_H.xpt.txt"))

blood_pressure_13_14_selected <- blood_pressure_13_14 %>%
    select(SEQN, BPQ020)


diabetes_13_14 <- read_xpt(file.path(path_to_data_13_14, "DIQ_H.xpt.txt"))

diabetes_13_14_selected <- diabetes_13_14 %>%
    select(SEQN, DIQ010)


periodontal_13_14 <- read_xpt(file.path(path_to_data_13_14, "OHXPER_H.xpt.txt"))

selected_cols <- colnames(periodontal_13_14)[grepl("^OHX\\d{2}(PC|LA)[A-Z]$", colnames(periodontal_13_14))]

periodontal_13_14_selected <- periodontal_13_14 %>%
    select(SEQN, all_of(selected_cols))


albumin_13_14 <- read_xpt(file.path(path_to_data_13_14, "BIOPRO_H.xpt.txt"))

albumin_13_14_selected <- albumin_13_14 %>%
    select(SEQN, LBDSALSI)

w_h_13_14 <- read_xpt(file.path(path_to_data_13_14, "BMX_H.xpt.txt"))

w_h_13_14_selected <- w_h_13_14 %>%
    select(SEQN, BMXWT, BMXHT)

## Merge datasets without NA and missing values

In [ ]:
# Merge datasets demographics and intrinsic capacity data

datasets_09_10 <- list(
  demo_09_10_selected, alcohol_09_10_selected, smoking_09_10_selected, med_conditions_09_10_selected,
  blood_pressure_09_10_selected, diabetes_09_10_selected,
  albumin_09_10_selected, w_h_09_10_selected, periodontal_09_10_selected
)

datasets_11_12 <- list(
  demo_11_12_selected, alcohol_11_12_selected, smoking_11_12_selected, med_conditions_11_12_selected,
  blood_pressure_11_12_selected, diabetes_11_12_selected,
  albumin_11_12_selected, w_h_11_12_selected, periodontal_11_12_selected
)

datasets_13_14 <- list(
  demo_13_14_selected, alcohol_13_14_selected, smoking_13_14_selected, med_conditions_13_14_selected,
  blood_pressure_13_14_selected, diabetes_13_14_selected,
  albumin_13_14_selected, w_h_13_14_selected, periodontal_13_14_selected
)

# Horizontal union for period 2009/10, 2011/12, 2013/14

df_09_10 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), datasets_09_10)

df_11_12 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), datasets_11_12)

df_13_14 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), datasets_13_14)

# Vertical union

df_final <- bind_rows(df_09_10, df_11_12, df_13_14)

print("Dimensions before removing NA values")
dim(df_final)

# Filter with AGE >= 60

df_final_age_60 <- subset(df_final, RIDAGEYR >= 60)

print("Dimensions with AGE >= 60")
dim(df_final_age_60)

# Excluding patients with missing values in features required for GNRI calculation (weight, height, albumin)

df_final_excluding_GNRI <- df_final_age_60[complete.cases(df_final_age_60[, c('BMXWT', 'BMXHT', 'LBDSALSI')]), ]

print("Dimensions without GNRI missing values")
dim(df_final_excluding_GNRI)

# Excluding patients with no examinations for Periodontitis Disease features

df_final_excluding_PD <- df_final_excluding_GNRI %>%
  filter(rowSums(!is.na(select(., starts_with("OHX")))) > 0)

print("Dimensions without PD missing values")
dim(df_final_excluding_PD)

# Excluding Edentulia

df_final_excluding_edentulus <- df_final_excluding_PD %>%
  filter(rowSums(select(., starts_with("OHX")) == 99, na.rm = FALSE) < ncol(select(., starts_with("OHX"))))

print("Dimensions without Edentulus patients")
dim(df_final_excluding_edentulus)

# Excluding patients with missing values in Confounding features

df_final_excluding_confounding <- df_final_excluding_edentulus[complete.cases(df_final_excluding_edentulus[, 
                                  c('RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'ALQ101', 'SMQ020', 'MCQ160B',
                                  'MCQ160C', 'MCQ160D', 'MCQ160E', 'MCQ160F', 'MCQ160L', 'MCQ220', 'BPQ020', 'DIQ010')]), ]

df_final_merged <- df_final_excluding_confounding %>%
  filter(!if_any(c(DMDEDUC2, ALQ101, MCQ160B, MCQ160C, MCQ160D,
                   MCQ160E, MCQ160F, MCQ160L, BPQ020, DIQ010), ~ . == 9))

print("Dimensions without Confounding missing values")
dim(df_final_merged)

In [ ]:
# Saving completed and cleaned dataframe for analysis

write.csv(df_final_merged, "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/merged_and_cleaned_df_09_14_GNRI_perio.csv", row.names = FALSE)

### Periodontitis Classification

In [ ]:
df_final_merged <- read.csv("/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/merged_and_cleaned_df_09_14_GNRI_perio.csv")

head(df_final_merged)

In [ ]:
classify_periodontitis <- function(df) {
  # Function to count sites on different teeth that meet a criterion
  count_sites_different_teeth <- function(tooth_sites,
                                          threshold, measurement_type) {
    teeth_with_sites <- list()
    for(tooth in unique(sub("(OHX\\d+).*", "\\1", names(tooth_sites)))) {
      # Include all interproximal sites
      # S = mesio-facial, D = distal, A = mesio-lingual, P = disto-lingual
      tooth_cols <- grep(paste0(tooth, measurement_type, "[SDAP]$"), 
                        names(tooth_sites), value = TRUE)
      values <- as.numeric(tooth_sites[tooth_cols])
      valid_values <- values[values != 99 & !is.na(values)]
      # If at least one site exceeds the threshold, add to list
      if(length(valid_values) > 0 && any(valid_values >= threshold)) {
        teeth_with_sites <- c(teeth_with_sites, tooth)
      }
    }
    # Returns the number of different teeth with sites exceeding the threshold
    return(length(teeth_with_sites))
  }

  ppd_cols <- grep("PC[SDAP]$", names(df), value = TRUE)
  cal_cols <- grep("LA[SDAP]$", names(df), value = TRUE)

  # Initializes the columns for classification
  df$severe <- FALSE
  df$moderate <- FALSE
  df$mild <- FALSE

  for(i in 1:nrow(df)) {
    ppd_values <- df[i, ppd_cols]
    cal_values <- df[i, cal_cols]

    # Counts teeth with sites that meet the criteria
    teeth_with_cal_6mm <- count_sites_different_teeth(cal_values, 6, "LA")
    teeth_with_cal_4mm <- count_sites_different_teeth(cal_values, 4, "LA")
    teeth_with_cal_3mm <- count_sites_different_teeth(cal_values, 3, "LA")
    teeth_with_ppd_5mm <- count_sites_different_teeth(ppd_values, 5, "PC")
    teeth_with_ppd_4mm <- count_sites_different_teeth(ppd_values, 4, "PC")

    # Severe periodontitis classification
    # ≥2 teeth with interproximal sites CAL ≥6 mm AND ≥1 tooth with PPD ≥5 mm
    if(teeth_with_cal_6mm >= 2 && teeth_with_ppd_5mm >= 1) {
      df$severe[i] <- TRUE
    }
    # Moderate periodontitis classification
    # ≥2 teeth with interproximal sites CAL ≥4 mm OR ≥2 tooth with PPD ≥5 mm
    else if(teeth_with_cal_4mm >= 2 || teeth_with_ppd_5mm >= 2) {
      df$moderate[i] <- TRUE
    }
    # Mild periodontitis
    # ≥2 teeth with interproximal sites CAL ≥3 mm AND
    # ≥2 teeth with interproximal sites PPD ≥4 mm
    else if(teeth_with_cal_3mm >= 2 &&
              teeth_with_ppd_4mm >= 2) {
      df$mild[i] <- TRUE
    }
  }

  # Final Classification
  df$periodontitis <- case_when(
    df$severe ~ "Severe",
    df$moderate ~ "Moderate",
    TRUE ~ "None/Mild"
    #df$mild ~ "Mild",
    #TRUE ~ "None"
  )

  return(df)
}

In [ ]:
df_classification <- classify_periodontitis(df_final_merged)

In [ ]:
table_counts <- table(df_classification$periodontitis)
table_percent <- prop.table(table_counts) * 100

print("Distribution of periodontitis categories (counting):")
print(table_counts)

print("Distribution of periodontitis categories (percentage):")
print(round(table_percent, 2))

In [ ]:
write.csv(df_classification, "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/perio_class_df_09_14_GNRI_perio.csv", row.names = FALSE)

### GNRI formula

In [ ]:
df_classification <- read.csv("/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/perio_class_df_09_14_GNRI_perio.csv")

head(df_classification)

In [ ]:
calculate_gnri <- function(df) {
  df$ideal_weight <- ifelse(
    df$RIAGENDR == 1,
    0.75 * df$BMXHT - 62.5,
    0.60 * df$BMXHT - 40
  )

  # Weight ratio (bounded at 1)
  weight_ratio <- pmin(df$BMXWT / df$ideal_weight, 1)

  # GNRI
  df$gnri_score <- (1.489 * df$LBDSALSI) + (41.7 * weight_ratio)

  # Multi-class category
  df$gnri_category <- cut(
    df$gnri_score,
    breaks = c(-Inf, 82, 91, 98, Inf),
    labels = c("Severe risk", "Moderate risk", "Low risk", "No risk"),
    right = TRUE
  )

  # Binary category
  df$gnri_binary <- ifelse(df$gnri_score < 98, "Low-GNRI", "High-GNRI")

  return(df)
}

In [ ]:
df_with_gnri <- calculate_gnri(df_classification)

head(df_with_gnri[c("ideal_weight", "gnri_score", "gnri_category", "gnri_binary")])

In [ ]:
table(df_with_gnri$gnri_binary)

In [ ]:
# Histogram and violin plot for GNRI score

options(repr.plot.width = 12, repr.plot.height = 6)

df_with_gnri$gender <- factor(df_with_gnri$RIAGENDR, levels = c(1, 2), labels = c("Male", "Female"))

hist_plot <- ggplot(df_with_gnri, aes(x = gnri_score)) +
  geom_histogram(binwidth = 5, fill = "#0073C2FF", color = "white", alpha = 0.8) +
  labs(title = "GNRI Score Distribution", x = "GNRI Score", y = "Count") +
  theme_minimal()

violin_plot <- ggplot(df_with_gnri, aes(x = gender, y = gnri_score, fill = gender)) +
  geom_violin(trim = FALSE, alpha = 0.6) +
  geom_boxplot(width = 0.1, outlier.shape = NA) +
  labs(title = "GNRI Score by Gender", x = "Gender", y = "GNRI Score") +
  scale_fill_manual(values = c("Male" = "#0073C2FF", "Female" = "#EFC000FF")) +
  theme_minimal()

hist_plot + violin_plot

In [ ]:
write.csv(df_with_gnri, "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/perio_gnri_df_09_14_GNRI_perio.csv", row.names = FALSE)

### Features selection

In [ ]:
df_with_gnri <- read.csv("/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/perio_gnri_df_09_14_GNRI_perio.csv")

head(df_with_gnri)

In [ ]:
# Features selection for the descriptive and regression analysis

columns_to_keep <- c("SEQN", "RIAGENDR", "RIDAGEYR", "RIDRETH1",
                     "DMDEDUC2", "INDFMPIR",
                     "SMQ020", "ALQ101", "MCQ160B", "MCQ160C",
                     "MCQ160D", "MCQ160E", "MCQ160F",
                     "MCQ160L", "MCQ220", "BPQ020",
                     "DIQ010", "LBDSALSI", "BMXWT", "BMXHT",
                     "periodontitis", "gnri_score", "gnri_category", "gnri_binary")
                     #"mean_ppd", "mean_cal")

df <- df_with_gnri[, columns_to_keep]

dim(df)
head(df)

In [ ]:
write.csv(df, "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/preprocessed_df_09_14_GNRI_perio.csv", row.names=FALSE)

### Loading DF & Weights

In [1]:
df <- read.csv("/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/preprocessed_df_09_14_GNRI_perio.csv")

head(df)

,SEQN,RIAGENDR,RIDAGEYR,RIDRETH1,DMDEDUC2,INDFMPIR,SMQ020,ALQ101,MCQ160B,MCQ160C,...,MCQ220,BPQ020,DIQ010,LBDSALSI,BMXWT,BMXHT,periodontitis,gnri_score,gnri_category,gnri_binary
,<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,...,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
1,51628,2,60,4,3,0.69,1,2,2,2,...,2,1,1,39,116.8,166.0,Moderate,99.771,No risk,High-GNRI
2,51633,1,80,3,4,1.27,1,1,2,2,...,2,2,2,43,79.1,174.3,Moderate,105.727,No risk,High-GNRI
3,51645,1,66,1,2,0.41,2,1,2,2,...,2,1,2,44,82.9,171.3,Severe,107.216,No risk,High-GNRI
4,51654,1,66,3,4,2.20,1,1,2,2,...,2,1,2,39,68.0,169.5,Moderate,99.771,No risk,High-GNRI
5,51661,2,60,1,3,2.75,2,1,2,2,...,2,2,2,40,73.5,151.4,None/Mild,101.260,No risk,High-GNRI
6,51680,2,60,4,4,2.59,1,1,2,2,...,2,2,2,39,98.9,176.6,Severe,99.771,No risk,High-GNRI


In [6]:
# Select weights

# Weights from Demographic datasets

demo_09_10 <- read_xpt(file.path(path_to_data_09_10, "DEMO_F.xpt"))

demo_09_10_weights <- demo_09_10 %>%
    select(SEQN, RIDAGEYR, WTMEC2YR, SDMVPSU, SDMVSTRA)

demo_11_12 <- read_xpt(file.path(path_to_data_11_12, "DEMO_G.xpt.txt"))

demo_11_12_weights <- demo_11_12 %>%
    select(SEQN, RIDAGEYR, WTMEC2YR, SDMVPSU, SDMVSTRA)

demo_13_14 <- read_xpt(file.path(path_to_data_13_14, "DEMO_H.xpt.txt"))

demo_13_14_weights <- demo_13_14 %>%
    select(SEQN, RIDAGEYR, WTMEC2YR, SDMVPSU, SDMVSTRA)

weights_09_10 <- list(
  demo_09_10_weights
)

weights_11_12 <- list(
  demo_11_12_weights
)

weights_13_14 <- list(
  demo_13_14_weights
)

# Horizontal union for period 2009/10, 2011/12, 2013/14

wt_09_10 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), weights_09_10)

wt_11_12 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), weights_11_12)

wt_13_14 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), weights_13_14)

# Vertical union

wt_final <- bind_rows(wt_09_10, wt_11_12, wt_13_14)

print("Dimensions before removing NA values")
dim(wt_final)

# Filter with AGE >= 60

wt_final_age_60 <- subset(wt_final, RIDAGEYR >= 60)

print("Dimensions with AGE >= 60")
dim(wt_final_age_60)

wt_final_age_60 <- subset(wt_final_age_60, select = -RIDAGEYR)
head(wt_final_age_60)

# Merge wt_final_age_60 with my final data frame

df_final_merged <- df %>%
  inner_join(wt_final_age_60, by = "SEQN")

dim(df_final_merged)

[1] "Dimensions before removing NA values"


[1] 30468     5

[1] "Dimensions with AGE >= 60"


[1] 5705    5

SEQN,WTMEC2YR,SDMVPSU,SDMVSTRA
<dbl>,<dbl>,<dbl>,<dbl>
51628,21000.339,2,75
51633,12381.115,1,77
51635,22502.507,1,79
51645,9590.458,1,75
51654,55670.350,2,86
51661,6385.327,2,88


[1] 2810   27

In [7]:
# Preprocessing for WTMEC2YR: divide it for the number of NHANES cycles used (3 for our case)

df_final_merged[, "wt"] = df_final_merged[, "WTMEC2YR"] / 3

head(df_final_merged)

,SEQN,RIAGENDR,RIDAGEYR,RIDRETH1,DMDEDUC2,INDFMPIR,SMQ020,ALQ101,MCQ160B,MCQ160C,...,BMXWT,BMXHT,periodontitis,gnri_score,gnri_category,gnri_binary,WTMEC2YR,SDMVPSU,SDMVSTRA,wt
,<dbl>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,...,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,51628,2,60,4,3,0.69,1,2,2,2,...,116.8,166.0,Moderate,99.771,No risk,High-GNRI,21000.339,2,75,7000.113
2,51633,1,80,3,4,1.27,1,1,2,2,...,79.1,174.3,Moderate,105.727,No risk,High-GNRI,12381.115,1,77,4127.038
3,51645,1,66,1,2,0.41,2,1,2,2,...,82.9,171.3,Severe,107.216,No risk,High-GNRI,9590.458,1,75,3196.819
4,51654,1,66,3,4,2.20,1,1,2,2,...,68.0,169.5,Moderate,99.771,No risk,High-GNRI,55670.350,2,86,18556.783
5,51661,2,60,1,3,2.75,2,1,2,2,...,73.5,151.4,None/Mild,101.260,No risk,High-GNRI,6385.327,2,88,2128.442
6,51680,2,60,4,4,2.59,1,1,2,2,...,98.9,176.6,Severe,99.771,No risk,High-GNRI,18341.270,1,79,6113.757


In [8]:
# Save df with weights

write.csv(df_final_merged, "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/final_preprocessed_df_09_14_GNRI_perio.csv", row.names=FALSE)

In [10]:
# Using srvyr to check population estimate
library(srvyr)

svy_obj <- df_final_merged %>%
  as_survey_design(
    ids = SDMVPSU,
    strata = SDMVSTRA,
    weights = wt,
    nest = TRUE
  )

pop_est <- svy_obj %>%
  summarize(pop = survey_total(1, vartype = "ci"))

print(pop_est)

# A tibble: 1 x 3
        pop   pop_low   pop_upp
      <dbl>     <dbl>     <dbl>
1 32653016. 29045087. 36260944.


### Descriptive Analysis

In [15]:
df <- read.csv("/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/final_preprocessed_df_09_14_GNRI_perio.csv")

head(df)

,SEQN,RIAGENDR,RIDAGEYR,RIDRETH1,DMDEDUC2,INDFMPIR,SMQ020,ALQ101,MCQ160B,MCQ160C,...,BMXWT,BMXHT,periodontitis,gnri_score,gnri_category,gnri_binary,WTMEC2YR,SDMVPSU,SDMVSTRA,wt
,<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,...,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<int>,<int>,<dbl>
1,51628,2,60,4,3,0.69,1,2,2,2,...,116.8,166.0,Moderate,99.771,No risk,High-GNRI,21000.339,2,75,7000.113
2,51633,1,80,3,4,1.27,1,1,2,2,...,79.1,174.3,Moderate,105.727,No risk,High-GNRI,12381.115,1,77,4127.038
3,51645,1,66,1,2,0.41,2,1,2,2,...,82.9,171.3,Severe,107.216,No risk,High-GNRI,9590.458,1,75,3196.819
4,51654,1,66,3,4,2.20,1,1,2,2,...,68.0,169.5,Moderate,99.771,No risk,High-GNRI,55670.350,2,86,18556.783
5,51661,2,60,1,3,2.75,2,1,2,2,...,73.5,151.4,None/Mild,101.260,No risk,High-GNRI,6385.327,2,88,2128.442
6,51680,2,60,4,4,2.59,1,1,2,2,...,98.9,176.6,Severe,99.771,No risk,High-GNRI,18341.270,1,79,6113.757


In [16]:
df$periodontitis <- factor(df$periodontitis, levels = c("None/Mild", "Moderate", "Severe"))

table(df$periodontitis)


None/Mild  Moderate    Severe 
     1057      1371       382 

In [21]:
# Recoding all features with consistent approach

df <- df %>%
  mutate(
      
    # Demographic variables
    RIAGENDR = factor(RIAGENDR, levels = c(1, 2),
                    labels = c("Male", "Female")),
    
    DMDEDUC2 = factor(DMDEDUC2, levels = 1:5,
                     labels = c("Less than 9th grade", "9-11th grade",
                                "High school graduate",
                                "Some college/AA degree",
                                "College graduate or above"),
                     ordered = TRUE),
    
    RIDRETH1 = factor(RIDRETH1, levels = 1:5,
                      labels = c("Mexican American", "Other Hispanic",
                                "Non-Hispanic White", "Non-Hispanic Black",
                                "Other Race")),
    
    # Lifestyle variables - Set "No" as reference
    SMQ020 = factor(SMQ020, levels = c(2, 1),
                   labels = c("No", "Yes")),  # Smoking status
    
    ALQ101 = factor(ALQ101, levels = c(2, 1),
                   labels = c("Under 12 drinks/1 yr", "Over 12 drinks/1 yr")),  # Alcohol
    
    # Medical conditions - all with "No" as reference level
    MCQ160B = factor(MCQ160B, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Heart failure
    
    MCQ160C = factor(MCQ160C, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Coronary heart disease
    
    MCQ160D = factor(MCQ160D, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Angina
    
    MCQ160E = factor(MCQ160E, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Heart attack
    
    MCQ160F = factor(MCQ160F, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Stroke
    
    MCQ220 = factor(MCQ220, levels = c(2, 1),
                   labels = c("No", "Yes")),  # Cancer
    
    MCQ160L = factor(MCQ160L, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Liver condition
    
    BPQ020 = factor(BPQ020, levels = c(2, 1),
                   labels = c("No", "Yes")),  # Hypertension
    
    # Multi-categorical - set "No" as reference
    DIQ010 = factor(DIQ010, levels = c(2, 1, 3),
                   labels = c("No", "Yes", "Borderline"))  # Diabetes
  )

# Verify transformations and check for any issues
summary(df_final_merged[, c("RIAGENDR", "DMDEDUC2", "RIDRETH1", "SMQ020", "ALQ101", 
               "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F", "MCQ220", "MCQ160L",
               "BPQ020", "DIQ010")])

   RIAGENDR                         DMDEDUC2                 RIDRETH1   
 Male  :1411   Less than 9th grade      :323   Mexican American  : 312  
 Female:1399   9-11th grade             :362   Other Hispanic    : 275  
               High school graduate     :627   Non-Hispanic White:1420  
               Some college/AA degree   :796   Non-Hispanic Black: 574  
               College graduate or above:702   Other Race        : 229  
  SMQ020                      ALQ101     MCQ160B    MCQ160C    MCQ160D   
 No  :1463   Under 12 drinks/1 yr: 906   No :2683   No :2608   No :2705  
 Yes :1345   Over 12 drinks/1 yr :1904   Yes: 127   Yes: 202   Yes: 105  
 NA's:   2                                                               
                                                                         
                                                                         
 MCQ160E    MCQ160F     MCQ220     MCQ160L    BPQ020            DIQ010    
 No :2622   No :2653   No  :2253   No :2684

In [22]:
# Create survey design for NHANES datasets

nhanes_design <- svydesign(
  id = ~SDMVPSU,
  strata = ~SDMVSTRA,
  weights = ~wt,
  nest = TRUE,
  data = df
)

In [ ]:
# Enhanced function for descriptive analysis with stratification
create_descriptive_table <- function(df, survey_design = NULL, stratify_by = NULL) {
  require(gtsummary)
  require(dplyr)
  require(survey)
  require(srvyr)
  
  # Check if survey design is provided
  use_survey_design <- !is.null(survey_design)
  use_stratification <- !is.null(stratify_by)
  
  # Normality test a priori for continuous variables
  continuous_vars <- c("RIDAGEYR", "INDFMPIR")
  
  normality_results <- list()
  
  for (var in continuous_vars) {
    # Limit: 5000 observations for Shapiro-Wilk test
    if (length(na.omit(df[[var]])) > 5000) {
      sample_data <- sample(na.omit(df[[var]]), 5000)
    } else {
      sample_data <- na.omit(df[[var]])
    }
    
    test_result <- shapiro.test(sample_data)
    normality_results[[var]] <- test_result$p.value > 0.05
    message(var, " p-value: ", test_result$p.value)
  }
  
  message("Normality test results:")
  for (var in names(normality_results)) {
    message(var, ": ", ifelse(normality_results[[var]], "Normal", "Non-normal"))
  }

  variables_to_include <- c("RIDAGEYR", "INDFMPIR", "RIAGENDR",
                "DMDEDUC2", "RIDRETH1", "SMQ020", "ALQ101",
                "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F", "MCQ220", "MCQ160L",
                "BPQ020", "DIQ010", "DLQ040_bin", "WHQ060_bin", "DLQ010_bin", "DLQ020_bin",
                "PFQ061I_bin", "DPQ050_bin", "DPQ020_bin", "DPQ010_bin")
  
  # Variables using median (IQR) or mean (SD)
  median_vars <- names(normality_results)[!unlist(normality_results)]

  stat_labels <- list(
    "RIDAGEYR" = "Age (mean, SD)",
    "INDFMPIR" = "Ratio of family income (mean, SD)",
    "RIAGENDR" = "Gender (n, %)",
    "RIDRETH1" = "Ethnicity (n, %)",
    "DMDEDUC2" = "Education (n, %)",
    "SMQ020" = "Smoking (n, %)",
    "ALQ101" = "Alcohol intake (n, %)",
    "MCQ160B" = "Heart Failure (n, %)",
    "MCQ160C" = "Coronary Heart (n, %)",
    "MCQ160D" = "Angina (n, %)",
    "MCQ160E" = "Heart Attack (n, %)",
    "MCQ160F" = "Stroke (n, %)",
    "MCQ220" = "Cancer (n, %)",
    "MCQ160L" = "Liver (n, %)",
    "BPQ020" = "Hypertension (n, %)",
    "DIQ010" = "Diabetes (n, %)",
    "DLQ040_bin" = "Cognition (n, %)",
    "WHQ060_bin" = "Weight loss (n, %)",
    "DLQ010_bin" = "Difficulty Hearing (n, %)",
    "DLQ020_bin" = "Difficulty Seeing (n, %)",
    "PFQ061I_bin" = "Locomotion (n, %)",
    "DPQ050_bin" = "Nutrition (n, %)",
    "DPQ020_bin" = "Mood (n, %)",
    "DPQ010_bin" = "Anhedonia (n, %)"
  )
  
  # Adjust labels based on normality
  for (var in names(normality_results)) {
    if (normality_results[[var]]) {
      stat_labels[[var]] <- gsub("\\(median, IQR\\)", "(mean, SD)", stat_labels[[var]])
    } else {
      stat_labels[[var]] <- gsub("\\(mean, SD\\)", "(median, IQR)", stat_labels[[var]])
    }
  }

  # Define statistics
  stat_list <- list(
    all_continuous() ~ "{mean} ({sd})",
    all_categorical() ~ "{n} ({p}%)"
  )
  for (var in median_vars) {
    stat_list[[var]] <- "{median} ({p25}, {p75})"
  }

  # Create base table with or without stratification
  if (use_stratification) {
    # Ensure stratification variable exists and convert to factor if needed
    if (!stratify_by %in% names(df)) {
      stop("Stratification variable '", stratify_by, "' not found in data")
    }
    
    # Convert to factor if not already
    if (!is.factor(df[[stratify_by]])) {
      df[[stratify_by]] <- as.factor(df[[stratify_by]])
    }
    
    table_strat <- df %>%
      tbl_summary(
        include = all_of(variables_to_include),
        by = all_of(stratify_by),
        statistic = stat_list,
        label = stat_labels,
        missing = "ifany",
        missing_text = "Missing",
        digits = all_continuous() ~ 2,
        value = all_categorical() ~ "level",
        type = all_categorical() ~ "categorical"
      ) %>%
      add_overall(col_label = "**Overall**", last = TRUE) %>%
      add_p(test = list(
        all_continuous() ~ "kruskal.test",  # Use non-parametric tests by default
        all_categorical() ~ "chisq.test"
      )) %>%
      modify_header(label = "**Characteristics**") %>%
      bold_labels()
    
  } else {
    table_strat <- df %>%
      tbl_summary(
        include = all_of(variables_to_include),
        statistic = stat_list,
        label = stat_labels,
        missing = "ifany",
        missing_text = "Missing",
        digits = all_continuous() ~ 2,
        value = all_categorical() ~ "level",
        type = all_categorical() ~ "categorical"
      ) %>%
      modify_header(label = "**Characteristics**") %>%
      bold_labels()
  }
  
  # Add weighted N column if survey design is provided
  if (use_survey_design) {
    # Calculate total weighted population
    survey_obj <- survey_design %>% 
      as_survey(options = list(lonely.psu = "adjust"))

    total_pop_in_millions <- survey_obj %>%
      summarize(pop = survey_total(1)) %>%
      mutate(pop_millions = pop/1000000) %>%
      pull(pop_millions)
    
    # Add weighted N column
    table_strat <- table_strat %>%
      modify_table_body(
        ~.x %>%
          dplyr::mutate(
            weighted_n = case_when(
              is.na(row_type) ~ "", 
              row_type == "label" ~ "",
              TRUE ~ ""
            )
          ) %>%
          dplyr::relocate(weighted_n, .after = label)
      ) %>%
      modify_header(
        weighted_n = "**Weighted N**\n**(Millions)**"
      )
    
    # Function to calculate weighted population for stratified analysis
    if (use_stratification) {
      calculate_weighted_pop_strat <- function(var_name, level = NULL, strat_level = NULL) {
        tryCatch({
          survey_data <- survey_design$variables
          
          if (is.null(level)) {
            # For continuous variables
            if (is.null(strat_level)) {
              # Overall
              var_data <- survey_data[[var_name]]
              weights_sum <- sum(weights(survey_design, "analysis")[!is.na(var_data)]) / 1000000
            } else {
              # By stratum
              var_data <- survey_data[[var_name]]
              strat_data <- survey_data[[stratify_by]]
              condition <- !is.na(var_data) & strat_data == strat_level & !is.na(strat_data)
              weights_sum <- sum(weights(survey_design, "analysis")[condition]) / 1000000
            }
          } else {
            # For categorical variables
            if (is.null(strat_level)) {
              # Overall
              var_data <- survey_data[[var_name]]
              condition <- var_data == level & !is.na(var_data)
              weights_sum <- sum(weights(survey_design, "analysis")[condition]) / 1000000
            } else {
              # By stratum
              var_data <- survey_data[[var_name]]
              strat_data <- survey_data[[stratify_by]]
              condition <- var_data == level & !is.na(var_data) & 
                          strat_data == strat_level & !is.na(strat_data)
              weights_sum <- sum(weights(survey_design, "analysis")[condition]) / 1000000
            }
          }
          return(weights_sum)
        }, error = function(e) {
          return(NA)
        })
      }
      
      # Get stratum levels
      strat_levels <- levels(df[[stratify_by]])
      
      # Update table with weighted populations
      table_strat$table_body <- table_strat$table_body %>%
        rowwise() %>%
        mutate(
          weighted_n = case_when(
            !is.na(row_type) & row_type == "label" & !is.na(variable) ~ 
              sprintf("%.2f", calculate_weighted_pop_strat(variable)),
            !is.na(row_type) & row_type == "level" & !is.na(variable) & !is.na(label) ~
              sprintf("%.2f", calculate_weighted_pop_strat(variable, label)),
            TRUE ~ weighted_n
          )
        ) %>%
        ungroup()
      
    } else {
      # Non-stratified weighted population calculation (your existing code)
      calculate_weighted_pop <- function(var_name, level = NULL) {
        tryCatch({
          if (is.null(level)) {
            var_data <- survey_design$variables[[var_name]]
            weights_sum <- sum(weights(survey_design, "analysis")[!is.na(var_data)]) / 1000000
            return(weights_sum)
          } else {
            var_data <- survey_design$variables[[var_name]]
            level_match <- var_data == level & !is.na(var_data)
            weights_sum <- sum(weights(survey_design, "analysis")[level_match]) / 1000000
            return(weights_sum)
          }
        }, error = function(e) {
          return(NA)
        })
      }
      
      table_strat$table_body <- table_strat$table_body %>%
        rowwise() %>%
        mutate(
          weighted_n = case_when(
            !is.na(row_type) & row_type == "label" & !is.na(variable) ~ 
              sprintf("%.2f", calculate_weighted_pop(variable)),
            !is.na(row_type) & row_type == "level" & !is.na(variable) & !is.na(label) ~
              sprintf("%.2f", calculate_weighted_pop(variable, label)),
            TRUE ~ weighted_n
          )
        ) %>%
        ungroup()
    }
  }
  
  # Add footnotes
  table_strat <- table_strat %>%
    modify_footnote(
      update = all_stat_cols() ~ "Values are n (%) for categorical variables, median (IQR) for non-normally distributed continuous variables, and mean (SD) for normally distributed continuous variables."
    )
  
  if (use_survey_design) {
    table_strat <- table_strat %>%
      modify_footnote(
        add = "Weighted N in millions represents the estimated US population based on NHANES survey weights."
      )
  }
  
  if (use_stratification) {
    table_strat <- table_strat %>%
      modify_footnote(
        add = "P-values from Kruskal-Wallis test for continuous variables and Chi-square test for categorical variables."
      )
  }
  
  return(table_strat)
}

# Function to create tables for each oral health feature
create_oral_health_tables <- function(df, survey_design = NULL) {
  
  oral_health_vars <- c("OHQ835_bin", "OHQ850_bin", "OHQ855_bin", "OHQ860_bin", "OHQ865_bin", "OHQ870_bin", "OHQ875_bin")
  oral_health_labels <- c(
    "OHQ835_bin" = "Do you think you might have gum disease?",
    #"OHQ845_bin" = "Rate the health of your teeth and gums",
    "OHQ850_bin" = "Ever had treatment for gum disease?",
    "OHQ855_bin" = "Any teeth became loose without an injury",
    "OHQ860_bin" = "Ever been told of bone loss around teeth",
    "OHQ865_bin" = "Noticed a tooth that doesn't look right",
    "OHQ870_bin" = "How many days use dental floss/device",
    "OHQ875_bin" = "Days used mouthwash for dental problem"
  )
  
  tables <- list()
  
  for (var in oral_health_vars) {
    message("Creating table for: ", oral_health_labels[[var]])
    
    tables[[var]] <- create_descriptive_table(
      df = df, 
      survey_design = survey_design, 
      stratify_by = var
    ) %>%
    modify_caption(paste("Participant characteristics by", oral_health_labels[[var]]))
  }
  
  return(tables)
}

# Function to save all tables
save_oral_health_tables <- function(tables, base_path) {
  
  oral_health_labels <- c(
    "OHQ835_bin" = "think_to_have_gum_disease",
    #"OHQ845_bin" = "rate_health_teeth_gum",
    "OHQ850_bin" = "treatment_gum_disease",
    "OHQ855_bin" = "loose_teeth_without_injury",
    "OHQ860_bin" = "bone_loss_around_teeth",
    "OHQ865_bin" = "tooth_not_look_right",
    "OHQ870_bin" = "dental_floss_use",
    "OHQ875_bin" = "mouthwash_use"
  )
  
  for (var in names(tables)) {
    file_name <- paste0(base_path, "_", oral_health_labels[[var]], ".docx")
    
    flex_table <- tables[[var]] %>% as_flex_table()
    
    save_as_docx(flex_table, path = file_name)
    
    message("Saved: ", file_name)
  }
}

In [ ]:
all_oral_tables <- create_oral_health_tables(
  df = df_final_merged,
  survey_design = nhanes_design
)

save_oral_health_tables(
  tables = all_oral_tables, 
  base_path = "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/results/NHANES_09_14_oral_health/Final_descriptive/7_oral_features/descriptive_analysis"
)